# ⚽ Football Players Detection — Entraînement YOLOv8

Ce notebook couvre l'ensemble du pipeline :
1. Installation des dépendances
2. Chargement du dataset depuis Roboflow
3. Vérification des données
4. Entraînement YOLOv8
5. Évaluation
6. Export ONNX

**Classes détectées :** `ball`, `goalkeeper`, `player`, `referee`

> 💡 Exécutez sur GPU (Google Colab / Kaggle / machine locale avec CUDA) pour de meilleures performances.

## 1. Installation des dépendances

In [ ]:
!pip install ultralytics roboflow --quiet

import ultralytics
ultralytics.checks()  # Vérifie GPU, PyTorch, version YOLO

## 2. Chargement du dataset depuis Roboflow

In [ ]:
from roboflow import Roboflow
import os

# Remplacez par votre clé API Roboflow (https://app.roboflow.com/ > Settings > API Key)
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(1)
dataset = version.download("yolov8")

print(f"Dataset téléchargé dans : {dataset.location}")

## 3. Vérification du dataset

In [ ]:
import yaml
import os

DATA_YAML = os.path.join(dataset.location, "data.yaml")

with open(DATA_YAML, "r") as f:
    data_config = yaml.safe_load(f)

print("Configuration du dataset :")
print(f"  Nombre de classes : {data_config['nc']}")
print(f"  Classes           : {data_config['names']}")
print(f"  Train             : {data_config.get('train', 'N/A')}")
print(f"  Val               : {data_config.get('val', 'N/A')}")
print(f"  Test              : {data_config.get('test', 'N/A')}")

In [ ]:
# Comptage des images et labels par split
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset.location, split, "images")
    lbl_dir = os.path.join(dataset.location, split, "labels")
    if os.path.exists(img_dir):
        n_imgs = len(os.listdir(img_dir))
        n_lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
        print(f"  {split:6s} → {n_imgs} images, {n_lbls} labels")

In [ ]:
# Visualisation d'un échantillon d'images annotées
import random
import cv2
import matplotlib.pyplot as plt
import numpy as np

COLORS = {
    0: (255, 215, 0),   # ball       → gold
    1: (0, 255, 0),     # goalkeeper → green
    2: (0, 120, 255),   # player     → blue
    3: (255, 0, 0),     # referee    → red
}
CLASS_NAMES = data_config["names"]

def draw_yolo_labels(img_path, lbl_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                cls, cx, cy, bw, bh = map(float, line.split())
                cls = int(cls)
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)
                color = COLORS.get(cls, (200, 200, 200))
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, CLASS_NAMES[cls], (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

train_img_dir = os.path.join(dataset.location, "train", "images")
train_lbl_dir = os.path.join(dataset.location, "train", "labels")
sample_imgs = random.sample(os.listdir(train_img_dir), min(6, len(os.listdir(train_img_dir))))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, fname in zip(axes.flat, sample_imgs):
    img_path = os.path.join(train_img_dir, fname)
    lbl_path = os.path.join(train_lbl_dir, fname.rsplit(".", 1)[0] + ".txt")
    ax.imshow(draw_yolo_labels(img_path, lbl_path))
    ax.set_title(fname, fontsize=8)
    ax.axis("off")
plt.suptitle("Échantillon du dataset (annotations YOLO)", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Configuration de l'entraînement

In [ ]:
# ─── Hyperparamètres ───────────────────────────────────────────────────────────
MODEL_SIZE   = "yolov8x"   # Options : yolov8n | yolov8s | yolov8m | yolov8l | yolov8x
                           #   n = nano (rapide)  ↔  x = extra-large (plus précis)
EPOCHS       = 100         # Nombre d'epochs (50-100 recommandé pour fine-tuning)
IMG_SIZE     = 640         # Résolution d'entraînement (640 standard YOLO)
BATCH_SIZE   = 16          # Réduire à 8 si OOM GPU
PATIENCE     = 20          # Early stopping : arrêt si pas d'amélioration après N epochs
PROJECT_NAME = "football_detection"
RUN_NAME     = f"{MODEL_SIZE}_ep{EPOCHS}"
DEVICE       = 0           # 0 = premier GPU ; 'cpu' si pas de GPU

print(f"Modèle     : {MODEL_SIZE}.pt (pré-entraîné COCO)")
print(f"Epochs     : {EPOCHS}")
print(f"Image size : {IMG_SIZE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Device     : {DEVICE}")

## 5. Entraînement YOLOv8

In [ ]:
from ultralytics import YOLO

# Chargement du modèle pré-entraîné sur COCO (transfer learning)
model = YOLO(f"{MODEL_SIZE}.pt")

# Lancement de l'entraînement
results = model.train(
    data       = DATA_YAML,
    epochs     = EPOCHS,
    imgsz      = IMG_SIZE,
    batch      = BATCH_SIZE,
    patience   = PATIENCE,
    device     = DEVICE,
    project    = PROJECT_NAME,
    name       = RUN_NAME,
    pretrained = True,
    optimizer  = "AdamW",
    lr0        = 0.001,
    lrf        = 0.01,
    mosaic     = 1.0,      # Augmentation mosaic
    mixup      = 0.1,      # Augmentation mixup
    copy_paste = 0.1,      # Augmentation copy-paste
    degrees    = 10.0,     # Rotation aléatoire
    fliplr     = 0.5,      # Flip horizontal
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    save       = True,
    plots      = True,
    verbose    = True,
)

## 6. Évaluation sur le set de validation

In [ ]:
import os

# Chemin vers le meilleur modèle sauvegardé
BEST_MODEL_PATH = os.path.join(PROJECT_NAME, RUN_NAME, "weights", "best.pt")
print(f"Meilleur modèle : {BEST_MODEL_PATH}")

# Chargement du meilleur modèle
best_model = YOLO(BEST_MODEL_PATH)

# Évaluation sur le set de validation
metrics = best_model.val(
    data   = DATA_YAML,
    imgsz  = IMG_SIZE,
    device = DEVICE,
)

print("\n─── Métriques de validation ───")
print(f"  mAP@50      : {metrics.box.map50:.4f}")
print(f"  mAP@50-95   : {metrics.box.map:.4f}")
print(f"  Précision   : {metrics.box.mp:.4f}")
print(f"  Rappel      : {metrics.box.mr:.4f}")

In [ ]:
# Affichage des courbes d'entraînement
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = os.path.join(PROJECT_NAME, RUN_NAME)
plot_files = ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
for ax, plot_file in zip(axes.flat, plot_files):
    path = os.path.join(run_dir, plot_file)
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
        ax.set_title(plot_file, fontsize=12)
    else:
        ax.set_title(f"{plot_file} (non trouvé)", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Inférence de test (aperçu visuel)

In [ ]:
# Test sur quelques images du set de test
test_img_dir = os.path.join(dataset.location, "test", "images")

if os.path.exists(test_img_dir):
    test_images = random.sample(os.listdir(test_img_dir), min(4, len(os.listdir(test_img_dir))))
    test_paths = [os.path.join(test_img_dir, f) for f in test_images]

    preds = best_model.predict(
        source    = test_paths,
        imgsz     = IMG_SIZE,
        conf      = 0.3,
        device    = DEVICE,
        save      = False,
    )

    fig, axes = plt.subplots(1, len(preds), figsize=(6 * len(preds), 6))
    if len(preds) == 1:
        axes = [axes]
    for ax, result in zip(axes, preds):
        img_annotated = result.plot()
        ax.imshow(cv2.cvtColor(img_annotated, cv2.COLOR_BGR2RGB))
        ax.axis("off")
    plt.suptitle("Prédictions sur le set de test", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Répertoire de test non trouvé.")

## 8. Export ONNX

In [ ]:
# Export du meilleur modèle au format ONNX
onnx_path = best_model.export(
    format     = "onnx",
    imgsz      = IMG_SIZE,
    opset      = 17,          # Opset ONNX (17 recommandé pour compatibilité maximale)
    simplify   = True,        # Simplifie le graphe ONNX (onnx-simplifier)
    dynamic    = False,       # False = taille fixe (640x640), True = taille dynamique
    half       = False,       # True = FP16 (nécessite GPU compatible)
)

print(f"\n✅ Modèle ONNX exporté : {onnx_path}")

In [ ]:
# Vérification du modèle ONNX
import onnx

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

print("✅ Modèle ONNX valide")
print(f"   Opset version : {onnx_model.opset_import[0].version}")

# Informations sur les entrées/sorties
print("\n─── Entrées du modèle ONNX ───")
for inp in onnx_model.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"  {inp.name}: {shape}")

print("\n─── Sorties du modèle ONNX ───")
for out in onnx_model.graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"  {out.name}: {shape}")

In [ ]:
# Test d'inférence avec ONNX Runtime
import onnxruntime as ort
import numpy as np
import time

session = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
print(f"Provider actif : {session.get_providers()[0]}")

# Inférence de test avec une image aléatoire
dummy_input = np.random.rand(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
input_name = session.get_inputs()[0].name

# Warmup
_ = session.run(None, {input_name: dummy_input})

# Benchmark
N = 10
start = time.time()
for _ in range(N):
    outputs = session.run(None, {input_name: dummy_input})
elapsed = (time.time() - start) / N * 1000

print(f"\n✅ Inférence ONNX Runtime opérationnelle")
print(f"   Latence moyenne ({N} runs) : {elapsed:.1f} ms")
print(f"   Shape de sortie           : {outputs[0].shape}")

## 9. Sauvegarde des artefacts

In [ ]:
import shutil

OUTPUT_DIR = "../models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Copie du modèle PyTorch (.pt)
dest_pt   = os.path.join(OUTPUT_DIR, "best_yolov8.pt")
dest_onnx = os.path.join(OUTPUT_DIR, "best_yolov8.onnx")

shutil.copy2(BEST_MODEL_PATH, dest_pt)
shutil.copy2(onnx_path, dest_onnx)

print(f"✅ Modèle PyTorch sauvegardé : {dest_pt}")
print(f"✅ Modèle ONNX sauvegardé    : {dest_onnx}")
print(f"\nTaille fichiers :")
print(f"  .pt   : {os.path.getsize(dest_pt)  / 1e6:.1f} MB")
print(f"  .onnx : {os.path.getsize(dest_onnx) / 1e6:.1f} MB")

## Récapitulatif

| Étape | Résultat |
|-------|----------|
| Dataset | `football-players-detection-3zvbc` v1 (Roboflow) |
| Classes | ball, goalkeeper, player, referee (nc=4) |
| Modèle | YOLOv8x (pré-entraîné COCO) |
| Export | ONNX opset 17, résolution 640×640 |
| Usage  | Charger `best_yolov8.pt` dans `Tracker('models/best_yolov8.pt')` |

### Utilisation dans le projet principal

```python
# Dans main.py, remplacez :
tracker = Tracker('models/best.pt')
# par :
tracker = Tracker('models/best_yolov8.pt')   # PyTorch
# ou :
tracker = Tracker('models/best_yolov8.onnx') # ONNX (inférence plus rapide)
```